# Krylov Methods — Interactive Numerical Comparisons

Compare Steepest Descent, Conjugate Gradient, GMRES, and BiCGSTAB
on symmetric positive-definite, complex Hermitian, and
nonsymmetric linear systems.

Explore Lanczos tridiagonalization, Ritz approximations,
and Galerkin projections.

**Instructions:** Select the parameters and click an experiment
button to display its numerical results and convergence plots.


In [ ]:
from pathlib import Path
import sys
import time

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "gradient_methods").is_dir() and (candidate / "krylov_methods").is_dir():
        repo_root = candidate
        break
else:
    raise RuntimeError("Start Jupyter inside your repository checkout.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

from gradient_methods.steepest_descent_solver import solve_steepest_descent
from krylov_methods.conjugate_gradient_solver import solve_conjugate_gradient
from krylov_methods.gmres_solver.GMRES import gmres
from krylov_methods.bicgstab_solver import solve_bicgstab
from krylov_methods.symmetric_lanczos_solver import lanczos_tridiagonalize, lanczos_galerkin
from krylov_methods.notebooks.notebook_utils import (
    spd_problem, hermitian_pd_problem, nonsymmetric_problem,
    residual_plot, error_plot, ritz_spectrum_plot, ritz_residual_plot,
)



In [ ]:
spd_kind = widgets.Dropdown(
    options=[("Real SPD", "real"), ("Complex Hermitian PD", "complex")],
    description="System:")
spd_n = widgets.IntSlider(value=20, min=5, max=45, step=5, description="Dimension:")
spd_cond = widgets.Dropdown(options=[("10", 10.), ("100", 100.),
                                         ("1,000", 1000.)], value=100., description="Condition:")
spd_rtol = widgets.Dropdown(options=[("1e-6", 1e-6), ("1e-8", 1e-8),
                                          ("1e-10", 1e-10)], value=1e-8, description="rtol:")
spd_budget = widgets.IntSlider(value=3000, min=100, max=8000, step=100,
                               description="SD budget:")
spd_run = widgets.Button(description="Run comparison", button_style="primary")
spd_out = widgets.Output()
spd_plot_output = widgets.Output()

def run_spd(_=None):
    with spd_out:
        clear_output(wait=True)
        spd_plot_output.clear_output(wait=True)
        generator = hermitian_pd_problem if spd_kind.value == "complex" else spd_problem
        A, b, x_true = generator(n=spd_n.value, condition=spd_cond.value, seed=42)
        rtol, atol = spd_rtol.value, 1e-12
        threshold = max(atol, rtol * np.linalg.norm(b))
        print(f"Dimension={len(b)} | dtype={A.dtype} | condition={np.linalg.cond(A):.3e}")
        print(f"Hermitian error={np.linalg.norm(A-A.conj().T):.2e}")
        print(f"||b||={np.linalg.norm(b):.3e} | stopping threshold={threshold:.3e}")
        gmres_points = [(0, float(np.linalg.norm(b)))]
        methods = {
            "Steepest Descent": lambda: solve_steepest_descent(
                A, b, rtol=rtol, atol=atol, max_iter=spd_budget.value),
            "CG": lambda: solve_conjugate_gradient(
                A, b, rtol=rtol, atol=atol, max_iter=10*len(b)),
            "GMRES (unrestarted)": lambda: gmres(
                A, b, rtol=rtol, atol=atol, maxiter=len(b),
                callback=lambda i, true_norm, estimate: gmres_points.append((i, true_norm))),
            "BiCGSTAB": lambda: solve_bicgstab(
                A, b, rtol=rtol, atol=atol, max_iter=10*len(b)),
        }
        histories = {}
        for label, run in methods.items():
            started = time.perf_counter()
            result = run()
            elapsed = time.perf_counter()-started
            x = result.x if label.startswith("GMRES") else result.solution
            res = np.linalg.norm(b-A@x)
            err = np.linalg.norm(x-x_true)/np.linalg.norm(x_true)
            hist = (tuple(np.array(gmres_points).T) if label.startswith("GMRES")
                    else result.residual_norms)
            histories[label] = hist
            print(f"{label:22} converged={str(result.converged):5} "
                  f"iter={result.iterations:5} residual={res:.2e} "
                  f"rel_error={err:.2e} time={elapsed:.3f}s")
        with spd_plot_output:
            residual_plot(histories, threshold, f"True residuals — {spd_kind.label}", figsize=(4.1, 3.0))
        print("Note: GMRES callback points use their true Arnoldi iteration indices.")

spd_run.on_click(run_spd)


In [ ]:
ns_n = widgets.IntSlider(value=30, min=10, max=80, step=10, description="Dimension:")
ns_restart = widgets.IntSlider(value=10, min=3, max=30, step=1, description="Restart:")
ns_budget = widgets.IntSlider(value=200, min=30, max=500, step=10, description="Max steps:")
ns_rtol = widgets.Dropdown(options=[("1e-6", 1e-6), ("1e-8", 1e-8),
                                         ("1e-10", 1e-10)], value=1e-8, description="rtol:")
ns_run = widgets.Button(description="Run comparison", button_style="primary")
ns_out = widgets.Output()
ns_plot_output = widgets.Output()

def run_ns(_=None):
    with ns_out:
        clear_output(wait=True)
        ns_plot_output.clear_output(wait=True)
        A, b, x_true = nonsymmetric_problem(n=ns_n.value, seed=42)
        threshold = max(1e-12, ns_rtol.value * np.linalg.norm(b))
        print(f"Dimension={len(b)} | ||A-A^T||={np.linalg.norm(A-A.T):.3e}")
        print(f"Condition number={np.linalg.cond(A):.3e} | threshold={threshold:.3e}")
        full_points = [(0, float(np.linalg.norm(b)))]
        restarted_points = [(0, float(np.linalg.norm(b)))]
        methods = {
            "GMRES (full)": lambda: gmres(A, b, restart=None, maxiter=len(b),
                rtol=ns_rtol.value, atol=1e-12,
                callback=lambda i, true_norm, est: full_points.append((i, true_norm))),
            "GMRES (restarted)": lambda: gmres(A, b, restart=min(ns_restart.value, len(b)),
                maxiter=ns_budget.value, rtol=ns_rtol.value, atol=1e-12,
                callback=lambda i, true_norm, est: restarted_points.append((i, true_norm))),
            "BiCGSTAB": lambda: solve_bicgstab(A, b, max_iter=ns_budget.value,
                rtol=ns_rtol.value, atol=1e-12),
        }
        histories = {}
        for label, run in methods.items():
            started = time.perf_counter()
            result = run()
            elapsed = time.perf_counter()-started
            x = result.x if label.startswith("GMRES") else result.solution
            history = (tuple(np.array(full_points).T) if label == "GMRES (full)"
                       else tuple(np.array(restarted_points).T) if label == "GMRES (restarted)"
                       else result.residual_norms)
            histories[label] = history
            error = np.linalg.norm(x-x_true)/np.linalg.norm(x_true)
            cycles = f", cycles={result.cycles}" if label.startswith("GMRES") else ""
            print(f"{label:20} converged={str(result.converged):5} "
                  f"iter={result.iterations:4}{cycles} "
                  f"residual={np.linalg.norm(b-A@x):.2e} "
                  f"rel_error={error:.2e} time={elapsed:.3f}s")
        with ns_plot_output:
            residual_plot(histories, threshold, "Nonsymmetric-system convergence", figsize=(4.1, 3.0))

ns_run.on_click(run_ns)


In [ ]:
lan_n = widgets.IntSlider(value=20, min=5, max=50, step=5, description="Dimension:")
lan_m = widgets.IntSlider(value=10, min=2, max=50, step=1, description="Krylov m:")
lan_kind = widgets.Dropdown(options=[("Real SPD", "real"),
                                         ("Complex HPD", "complex")], description="System:")
lan_run = widgets.Button(description="Run Lanczos", button_style="primary")
lan_out = widgets.Output()
lan_spectrum_output = widgets.Output()
lan_residual_output = widgets.Output()

def run_lanczos(_=None):
    with lan_out:
        clear_output(wait=True)
        lan_spectrum_output.clear_output(wait=True)
        lan_residual_output.clear_output(wait=True)
        n = lan_n.value
        m = min(lan_m.value, n)
        generator = hermitian_pd_problem if lan_kind.value == "complex" else spd_problem
        A, b, x_true = generator(n=n, condition=100, seed=17)
        projection = lanczos_tridiagonalize(A, b, m=m, reorthogonalize=True)
        V, T = projection.basis, projection.tridiagonal
        ortho_error = np.linalg.norm(V.conj().T @ V - np.eye(V.shape[1]))
        projection_error = np.linalg.norm(V.conj().T @ A @ V - T)
        print(f"Dimension={n}, requested m={m}, effective m={projection.iterations}")
        print(f"Orthogonality error={ortho_error:.3e}")
        print(f"Projection error={projection_error:.3e}")
        print(f"Ritz range=[{projection.ritz_values[0]:.4g}, "
              f"{projection.ritz_values[-1]:.4g}]")
        with lan_spectrum_output:
            ritz_spectrum_plot(A, projection.ritz_values)
        with lan_residual_output:
            ritz_residual_plot(A, V, T)
        gal = lanczos_galerkin(A, b, m=m, tol=1e-9, reorthogonalize=True)
        if gal.solution is None:
            print("Galerkin projection unavailable:", gal.message)
        else:
            residual = np.linalg.norm(b-A@gal.solution)
            error = np.linalg.norm(gal.solution-x_true)/np.linalg.norm(x_true)
            print(f"Galerkin converged={gal.converged} | "
                  f"true residual={residual:.3e} | relative error={error:.3e}")

lan_run.on_click(run_lanczos)


In [ ]:

# -------------------------------------------------------
# Krylov Comparison — Compact Voilà dashboard
# -------------------------------------------------------

spd_status = widgets.HTML()
ns_status = widgets.HTML()
lan_status = widgets.HTML()

spd_parameters = (
    spd_kind, spd_n, spd_cond, spd_rtol, spd_budget
)

ns_parameters = (
    ns_n, ns_restart, ns_budget, ns_rtol
)

lan_parameters = (
    lan_kind, lan_n, lan_m
)

last_runs = {
    "spd": None,
    "ns": None,
    "lan": None,
}


# -------------------------------------------------------
# Simulation status indicators
# -------------------------------------------------------

def parameter_values(parameters):
    return tuple(widget.value for widget in parameters)


def update_status(name, parameters, indicator):
    previous = last_runs[name]

    if previous is None:
        indicator.value = ""
    elif parameter_values(parameters) == previous:
        indicator.value = (
            "<span style='color:#22863a;font-weight:600;'>"
            "● Simulation up to date</span>"
        )
    else:
        indicator.value = (
            "<span style='color:#b36b00;font-weight:600;'>"
            "● Parameters changed — rerun required</span>"
        )


def refresh_spd(change=None):
    update_status("spd", spd_parameters, spd_status)


def refresh_ns(change=None):
    update_status("ns", ns_parameters, ns_status)


def refresh_lan(change=None):
    update_status("lan", lan_parameters, lan_status)


for parameter in spd_parameters:
    parameter.observe(refresh_spd, names="value")

for parameter in ns_parameters:
    parameter.observe(refresh_ns, names="value")

for parameter in lan_parameters:
    parameter.observe(refresh_lan, names="value")


# -------------------------------------------------------
# Four independent plot panels
# -------------------------------------------------------

def plot_panel(title, plot_output):
    return widgets.VBox(
        [
            widgets.HTML(
                "<div style='font-weight:600;"
                "text-align:center;font-size:13px;'>"
                + title
                + "</div>"
            ),
            plot_output,
        ],
        layout=widgets.Layout(
            flex="1 1 240px",
            min_width="220px",
            max_width="100%",
            padding="4px",
            overflow="hidden",
        ),
    )


spd_plot_panel = plot_panel(
    "SPD / Hermitian PD Convergence",
    spd_plot_output,
)

ns_plot_panel = plot_panel(
    "Nonsymmetric Convergence",
    ns_plot_output,
)

lan_spectrum_panel = plot_panel(
    "Lanczos Ritz Spectrum",
    lan_spectrum_output,
)

lan_residual_panel = plot_panel(
    "Lanczos Ritz-Pair Residuals",
    lan_residual_output,
)

all_plot_panels = (
    spd_plot_panel,
    ns_plot_panel,
    lan_spectrum_panel,
    lan_residual_panel,
)

# Keep plots hidden until their experiments run.
for panel in all_plot_panels:
    panel.layout.display = "none"

plots_row = widgets.HBox(
    list(all_plot_panels),
    layout=widgets.Layout(
        width="100%",
        flex_flow="row wrap",
        gap="5px",
        align_items="flex-start",
    ),
)


# -------------------------------------------------------
# Expandable numerical reports
# -------------------------------------------------------

spd_details = widgets.Accordion(
    children=[spd_out]
)
spd_details.set_title(
    0, "SPD / Hermitian PD solver comparison"
)
spd_details.selected_index = None


ns_details = widgets.Accordion(
    children=[ns_out]
)
ns_details.set_title(
    0, "Nonsymmetric solver comparison"
)
ns_details.selected_index = None


lan_details = widgets.Accordion(
    children=[lan_out]
)
lan_details.set_title(
    0, "Lanczos numerical diagnostics"
)
lan_details.selected_index = None


for details in (
    spd_details,
    ns_details,
    lan_details,
):
    details.layout.display = "none"
    details.layout.flex = "1 1 300px"
    details.layout.min_width = "240px"


# -------------------------------------------------------
# Reveal results after execution
# -------------------------------------------------------

def show_spd_results(change=None):
    spd_plot_panel.layout.display = ""
    spd_details.layout.display = ""

    last_runs["spd"] = parameter_values(
        spd_parameters
    )

    refresh_spd()


def show_ns_results(change=None):
    ns_plot_panel.layout.display = ""
    ns_details.layout.display = ""

    last_runs["ns"] = parameter_values(
        ns_parameters
    )

    refresh_ns()


def show_lan_results(change=None):
    lan_spectrum_panel.layout.display = ""
    lan_residual_panel.layout.display = ""
    lan_details.layout.display = ""

    last_runs["lan"] = parameter_values(
        lan_parameters
    )

    refresh_lan()


# Preserve the original numerical callbacks.
# Register presentation callbacks after them.
spd_run.on_click(show_spd_results)
ns_run.on_click(show_ns_results)
lan_run.on_click(show_lan_results)


# -------------------------------------------------------
# Experiment control panels
# -------------------------------------------------------

spd_controls = widgets.VBox(
    [
        widgets.HTML(
            "<h3>SPD / Hermitian PD Comparison</h3>"
        ),
        spd_kind,
        spd_n,
        spd_cond,
        spd_rtol,
        spd_budget,
        spd_run,
        spd_status,
    ],
    layout=widgets.Layout(
        flex="1 1 350px",
        min_width="280px",
        padding="10px",
        border="1px solid #dddddd",
    ),
)

ns_controls = widgets.VBox(
    [
        widgets.HTML(
            "<h3>Nonsymmetric Comparison</h3>"
        ),
        ns_n,
        ns_restart,
        ns_budget,
        ns_rtol,
        ns_run,
        ns_status,
    ],
    layout=widgets.Layout(
        flex="1 1 350px",
        min_width="280px",
        padding="10px",
        border="1px solid #dddddd",
    ),
)

lan_controls = widgets.VBox(
    [
        widgets.HTML(
            "<h3>Lanczos Analysis</h3>"
        ),
        lan_kind,
        lan_n,
        lan_m,
        lan_run,
        lan_status,
    ],
    layout=widgets.Layout(
        flex="1 1 350px",
        min_width="280px",
        padding="10px",
        border="1px solid #dddddd",
    ),
)

controls_row = widgets.HBox(
    [
        spd_controls,
        ns_controls,
        lan_controls,
    ],
    layout=widgets.Layout(
        width="100%",
        flex_flow="row wrap",
        gap="10px",
        align_items="stretch",
    ),
)

details_row = widgets.HBox(
    [
        spd_details,
        ns_details,
        lan_details,
    ],
    layout=widgets.Layout(
        width="100%",
        flex_flow="row wrap",
        gap="10px",
        align_items="flex-start",
    ),
)


# -------------------------------------------------------
# Final interactive dashboard
# -------------------------------------------------------

krylov_dashboard = widgets.VBox(
    [
        widgets.HTML("""
        <style>
        .krylov-compact-dashboard .output_area img,
        .krylov-compact-dashboard .jp-OutputArea-output img {
            max-width:100% !important;
            height:auto !important;
        }
        </style>
        """),
        controls_row,
        plots_row,
        details_row,
    ],
    layout=widgets.Layout(
        width="100%",
        gap="12px",
    ),
)

krylov_dashboard.add_class(
    "krylov-compact-dashboard"
)

display(krylov_dashboard)

# No automatic simulations.
# Plots and reports appear after experiment buttons are clicked.
